# European Option Pricing with Continuous Dividends (Black-Scholes-Merton)

This notebook fetches 1 year of data for **KO (Coca-Cola)** using `yfinance`, computes the annualized historical volatility, estimates the **continuous dividend yield $q$** from KO's real dividend payments, and prices both European Call and Put options with the **Black-Scholes-Merton model with dividends**.

**Plan:**
1. fetch KO data + risk-free rate
2. historical volatility from log returns
3. calculate the dividend yield $q$
4. price Call & Put with the dividend formula + check put-call parity
5. implied volatility with Brent's method

**What changes vs plain Black-Scholes:** a dividend-paying stock grows slower than $r$ in the risk-neutral world - the drift becomes $r - q$.

## Inputs

The Black-Scholes-Merton model prices a European option from **6 inputs**:

| Input | Symbol | Where we get it |
|---|---|---|
| Current stock price | $S_0$ | Market (yfinance, latest close) |
| Strike price | $K$ | Option contract |
| Time to maturity | $T$ | Expiry date - today (in years) |
| Risk-free rate | $r$ | Treasury bill yield (`^IRX`) |
| Volatility | $\sigma$ | Historical log returns of the stock |
| **Dividend yield** | $q$ | KO's actual dividend payments |

The model (Merton, 1973) keeps all Black-Scholes assumptions (no arbitrage, constant volatility, continuous trading) and adds one: **the stock pays dividends continuously at rate $q$ per year**. This single change is what modifies the formula.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import scipy.stats as stats
from scipy.optimize import brentq

# Fetch 1 year of historical daily data for KO (a dividend-paying stock)
ticker_symbol = "KO"
df = yf.Ticker(ticker_symbol).history(period="1y")

# Extract the most recent closing price as the current stock price (S0)
S0 = df['Close'].iloc[-1]
print(f"Current {ticker_symbol} Stock Price (S0): ${S0:.2f}")

Current KO Stock Price (S0): $93.90


## Volatility from historical data

The model needs $\sigma$ = **annualized volatility** of the stock's log returns.

Daily log return:

$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns add up over time, so their standard deviation grows with the square root of time. With 252 trading days in a year:

$$\boxed{\sigma_{\text{annual}} = \operatorname{std}(r_t)\times\sqrt{252}}$$

**Assumption:** log returns are (roughly) normally distributed with constant volatility - then the stock price is lognormal, which is exactly what Black-Scholes assumes.

In [2]:
# Calculate daily log returns: ln(P_t / P_t-1)
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Drop missing values if any
log_returns = df['Log_Return'].dropna()

# standard deviation and annualization
daily_volatility = log_returns.std()
historical_volatility = daily_volatility * np.sqrt(252)

print(f"Daily Volatility: {daily_volatility:.6f}")
print(f"Annualized Historical Volatility (sigma): {historical_volatility:.4f} ({historical_volatility * 100:.2f}%)")

Daily Volatility: 0.011934
Annualized Historical Volatility (sigma): 0.1894 (18.94%)


## How to calculate the continuous dividend yield $q$

**In simple words:** a continuous dividend yield $q$ means *"every year the stock pays out q% of its price as dividends, spread evenly over the year"*. Real companies pay a few big checks 4 times a year, but for pricing we **smooth them out into a continuous stream** - this is the standard trick, and it works well for stocks and indexes that pay regularly.

### Three ways to calculate $q$ from real data

1. **Trailing yield (what we use below)** - add up the dividends paid in the last 12 months and divide by today's price:

$$q = \frac{\sum \text{dividends paid in the last 12 months}}{S_0}$$

2. **Reported yield** - Yahoo Finance already publishes an annual dividend yield (`Ticker.info['dividendYield']`); just convert percent to decimal: $q = \text{yield}/100$.

3. **Forward yield** - if you know what the company will pay over the next year:

$$q = \frac{\text{expected dividends next 12 months}}{S_0}$$

### Why $q$ shows up in the formula as $e^{-qT}$

- Over $T$ years, a stock paying continuous yield $q$ hands a factor $e^{-qT}$ of its value to shareholders, so only $\;Se^{-qT}$ of it is still yours at expiry.
- The **present value of all dividends** paid over the option's life:

$$S_0\left(1 - e^{-qT}\right)$$

- The risk-neutral growth of the stock simply changes: $\;r \rightarrow r - q$.

**Example:** with $S_0 = 70$ and $q = 0.03$ (3%/year), the dividends over 1 year are worth about $70 \times (1 - e^{-0.03}) \approx \$2.07$.

In [4]:
# -----------------------------------------------------------------
# Calculate the continuous dividend yield q
# Method 1 (used below): trailing yield = dividends last 12 months / price
# Method 2 (cross-check): yield reported by Yahoo Finance
# -----------------------------------------------------------------
tk = yf.Ticker(ticker_symbol)
div_series = tk.dividends                       # past dividend payments (ex-date, amount)

if len(div_series) > 0:
    if div_series.index.tz is not None:
        div_series.index = div_series.index.tz_localize(None)

    last_12m = div_series[div_series.index >= (datetime.now() - pd.Timedelta(days=365))]
    total_divs_12m = float(last_12m.sum())
    q_trailing = total_divs_12m / S0

    print("Dividend payments in the last 12 months:")
    for date, amt in last_12m.items():
        print(f"   {date.date()}   ${amt:.4f}")
    print(f"\nTotal dividends (12m) : ${total_divs_12m:.4f}")
    print(f"q (trailing) = total / S0 = {total_divs_12m:.4f} / {S0:.2f} = {q_trailing:.4f} ({q_trailing * 100:.2f}%)")
else:
    q_trailing = np.nan
    print(f"WARNING: no dividend history found for {ticker_symbol}")

# Method 2: annual yield reported by Yahoo Finance (cross-check)
try:
    dy = tk.info.get('dividendYield')
    q_reported = (dy / 100) if (dy and dy > 1) else float(dy or np.nan)
except Exception:
    q_reported = np.nan

# use the trailing yield if available, otherwise the reported one
if np.isfinite(q_trailing) and q_trailing > 0:
    q = q_trailing
elif np.isfinite(q_reported) and q_reported > 0:
    q = q_reported
else:
    q = 0.02                                   # otherwise default = (2%)

print(f"\nyfinance reported dividend yield : {q_reported * 100:.2f}%")
print(f"Continuous dividend yield used q : {q:.4f} ({q * 100:.2f}%)")

Dividend payments in the last 12 months:
   2025-12-01   $0.5100
   2026-03-13   $0.5300
   2026-06-15   $0.5300
   2026-09-15   $0.5300

Total dividends (12m) : $2.1000
q (trailing) = total / S0 = 2.1000 / 93.90 = 0.0224 (2.24%)

yfinance reported dividend yield : 2.26%
Continuous dividend yield used q : 0.0224 (2.24%)


## Option parameters and the $d_1$, $d_2$ terms (dividend version)

- **Strike** $K$ - we pick the nearest-dollar strike, so the option starts at-the-money
- **Time to maturity** $T$ - years until expiry: $T = \dfrac{\text{days to expiry}}{365.25}$
- **Risk-free rate** $r$ - continuously compounded (here from the 13-week T-bill index `^IRX`)
- **Dividend yield** $q$ - just calculated above

Everything is identical to plain Black-Scholes **except that the drift $r$ becomes $r - q$**:

$$d_1 = \frac{\ln(S_0/K) + \left(r - q + \tfrac{1}{2}\sigma^2\right)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

- $N(d_1)$ - how much of the stock you effectively "keep", chance-weighted for drift
- $N(d_2)$ - risk-neutral probability that the option expires in the money

In [5]:
# Define option parameters

K = round(S0)                # nearest-dollar strike = at the money
expiry_date = datetime(2027, 6, 18)
today = datetime.now()

# ----------------------------------------------------------------------- #

# time to maturity (T) in years
T = (expiry_date - today).days / 365.25
print(f"Strike Price (K)          : ${K}")
print(f"Time to Maturity (T)      : {T:.8f} years ({(expiry_date - today).days} days)")

# Risk-Free Rate from ^IRX and historical volatility:
irx = yf.Ticker("^IRX")
# recent close price
r = irx.history(period="5d")['Close'].iloc[-1] / 100
sigma = historical_volatility
print(f"Risk-Free Rate (r)        : {r:.4f} ({r * 100:.2f}%)")
print(f"Dividend Yield (q)        : {q:.4f} ({q * 100:.2f}%)")
print(f"Volatility Used (sigma)   : {sigma:.4f} ({sigma * 100:.2f}%)")

# dividend-adjusted d1 and d2
d1 = (np.log(S0 / K) + (r - q + 0.5 * (sigma ** 2)) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

def black_scholes_call(S0, K, T, r, sigma):
    return S0 * np.exp(-q * T) * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def black_scholes_put(S0, K, T, r, sigma):
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * np.exp(-q * T) * stats.norm.cdf(-d1)

Strike Price (K)          : $94
Time to Maturity (T)      : 0.65982204 years (241 days)
Risk-Free Rate (r)        : 0.0408 (4.08%)
Dividend Yield (q)        : 0.0224 (2.24%)
Volatility Used (sigma)   : 0.1894 (18.94%)


## Black-Scholes-Merton pricing formulas (with dividend yield $q$)

**Call (right to buy):**
$$C = S_0 e^{-qT} N(d_1) - K e^{-rT} N(d_2)$$

**Put (right to sell):**
$$P = K e^{-rT} N(-d_2) - S_0 e^{-qT} N(-d_1)$$

**How to read it:** compared to plain Black-Scholes, every $S_0$ is replaced by $S_0e^{-qT}$ - the stock with the dividends "taken out". Everything else is identical.

**Equivalent view (forward price):** a stock with dividend yield $q$ has forward price $F = S_0e^{(r-q)T}$, and the model is just Black-Scholes applied to that forward:

$$C = e^{-rT}\left[F\,N(d_1) - K\,N(d_2)\right]$$

**Put-call parity with dividends** gets its own section right below (the printout checks it).

## Put-call parity with dividends (a free correctness check)

**In simple words:** a call and a put with the *same strike* and the *same expiry* are linked - one can be built from the other plus some cash/stock, adjusted for the dividends the stock pays before expiry.

**Why is it true?** Two portfolios that pay the **same amount at expiry** must cost the same today:

- **Portfolio A:** buy 1 call + keep $Ke^{-rT}$ cash in the bank. At expiry you end up with $\max(S_T, K)$ (exactly as without dividends).
- **Portfolio B:** buy 1 put + buy the stock - but the stock hands out dividends along the way, so only $S_0e^{-qT}$ of its value is still yours at expiry. You also end up with $\max(S_T, K)$.

Same payoff -> same price today:

$$C + Ke^{-rT} = P + S_0e^{-qT} \qquad\Longleftrightarrow\qquad \boxed{C - P = S_0e^{-qT} - Ke^{-rT}}$$

**Good to know:**

- Parity needs **no volatility and no model** - it follows from no-arbitrage alone, so it must hold *exactly*.
- The only difference vs the no-dividend version: the stock side shrinks from $S_0$ to $S_0e^{-qT}$, because the stock "loses" the dividends it pays before expiry.
- If a real market broke parity, you could buy the cheap side and sell the rich side for a guaranteed profit (arbitrage). That is why the printout below shows `C - P` next to the *should be* value - the residual must be ~0.

In [6]:
# option prices
call_price = black_scholes_call(S0, K, T, r, sigma)
put_price = black_scholes_put(S0, K, T, r, sigma)

# put-call parity check:  C - P  should equal  S0*e^(-qT) - K*e^(-rT)
parity_lhs = call_price - put_price
parity_rhs = S0 * np.exp(-q * T) - K * np.exp(-r * T)

print("=" * 45)
print(f"Underlying Asset   : {ticker_symbol}")
print(f"Current Stock Price: ${S0:.2f}")
print(f"Strike Price       : ${K}")
print(f"Expiration Date    : {expiry_date}    T = {T:.4f}")
print(f"Dividend Yield (q) : {q * 100:.2f}%")
print(f"Volatility Used    : {sigma * 100:.2f}%")
print("-" * 45)
print(f"Call Option Price  : ${call_price:.2f}")
print(f"Put Option Price   : ${put_price:.2f}")
print("-" * 45)
print(f"Put-call parity    : C - P = {parity_lhs:.4f}   (should be {parity_rhs:.4f})")
print(f"Parity residual    : {abs(parity_lhs - parity_rhs):.8f}")
print("=" * 45)

Underlying Asset   : KO
Current Stock Price: $93.90
Strike Price       : $94
Expiration Date    : 2027-06-18 00:00:00    T = 0.6598
Dividend Yield (q) : 2.24%
Volatility Used    : 18.94%
---------------------------------------------
Call Option Price  : $6.17
Put Option Price   : $5.15
---------------------------------------------
Put-call parity    : C - P = 1.0195   (should be 1.0195)
Parity residual    : 0.00000000


## Implied volatility (with dividends)

Same idea as without dividends: we know the *market price* and search for the $\sigma$ that would produce it:

$$\text{find } \sigma \text{ such that } BSQ(S_0, K, T, r, q, \sigma) = \text{market price}$$

The pricer used to reverse-engineer $\sigma$ must include $q$ - otherwise part of the dividend effect would be wrongly reported as "volatility". We solve it with **Brent's method** (`scipy.optimize.brentq`), a safe root-finder for

$$f(\sigma) = BSQ(\sigma) - \text{market price} = 0$$

The price always increases with $\sigma$, so the root is unique; if the market price lies outside the model's bounds, no root exists and we return `NaN`.

In [10]:
def bs_price(CP, S0, K, sigma, T, r, q):
    """Black-Scholes-Merton price with continuous dividend yield q."""
    if sigma <= 0:
        return 0.0
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if CP.upper() == 'CALL':
        price = stats.norm.cdf(d1) * S0 * np.exp(-q * T) - stats.norm.cdf(d2) * K * np.exp(-r * T)
    else:
        price = stats.norm.cdf(-d2) * K * np.exp(-r * T) - stats.norm.cdf(-d1) * S0 * np.exp(-q * T)
    return price

def implied_volatility(market_price, CP, S0, K, T, r, q):
    """
    Finds the implied volatility by reverse-engineering the Black-Scholes-Merton price
    using Brent's root-finding method.
    """
    # objective function: model price - market price
    def objective_function(sigma):
        return bs_price(CP, S0, K, sigma, T, r, q) - market_price

    try:
        # search for volatility between 0.001 (0.1%) and 5.0 (500%)
        iv = brentq(objective_function, 1e-3, 5.0)
        return iv
    except ValueError:
        # returns NaN if the market price is invalid
        return np.nan

market_price_call = 4.50    # observed market price of the call option
market_price_put = 3.50

iv_call = implied_volatility(market_price_call, 'CALL', S0, K, T, r, q)
iv_put = implied_volatility(market_price_put, 'PUT', S0, K, T, r, q)

print()
print(f"Market prices used : call ${market_price_call:.2f}, put ${market_price_put:.2f}")
print()
print(f"Implied Volatility_CALL: {iv_call:.4f} ({iv_call * 100:.2f}%)")
print()
print(f"Implied Volatility_PUT: {iv_put:.4f} ({iv_put * 100:.2f}%)")


Market prices used : call $4.50, put $3.50

Implied Volatility_CALL: 0.1332 (13.32%)

Implied Volatility_PUT: 0.1338 (13.38%)
